# TFM 1 · Run the TFM vs GP replay benchmark
Predicts each output one step ahead on the real BO runs, for GPs and TFMs, then builds the three report pages.
Code: `notebooks/tfm_bench/`. Data: `data/`. Everything produced: `results/tfm_bench/` (both kept out of git).

**Colab:** Runtime → Change runtime type → **T4 GPU**. Secrets (key icon, "Notebook access" on): `GITHUB_TOKEN`
(private repo), `TABPFN_TOKEN` (TabPFN 3.5), optional `HF_TOKEN`. `data/` and `results/` live in `DRIVE_ROOT` on Drive,
so a disconnect loses at most 25 steps: rerun the cells, finished work is skipped, interrupted work resumes.

**Local Jupyter:** open this file from `notebooks/`; set the same names as environment variables.

In [ ]:
# ---- settings ------------------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USER/ASML.git"   # <- your repo (Colab only; locally the notebook runs in place)
DRIVE_ROOT = "/content/drive/MyDrive/ASML"           # Colab only: Drive folder holding data/ and results/
STATE_ZIP = "../results/tfm_bench/benchmark_state_slim.zip"   # restored once if data/ has no opt_results CSVs; "" = none
DATASETS = "all"          # "all" or "20261001_161410,20261002_081740"
TFMS = "all"              # "all" = causilo,kumo,kumo_medium,kumo_small,tabpfn2,tabpfn35 ; or e.g. "tabpfn35,kumo"
ARMS = "all"              # "all" = ctx0,ctx5,ctxall
GPS = "all"               # "all" = gpr_laplace,gpr_botorch ; "" = skip the GPs
OUTPUTS = ["dy"]          # "dy" (die yield) and/or "EUV_mJ", "Rx_deg", "Ry_deg", "TargetSize_um"
MAX_STEPS = None          # e.g. 20 for a quick test (writes real result folders!)

In [ ]:
# Setup. Works in Colab (clones the repo, links data/ + results/ to Drive) and in local Jupyter (runs in place).
# Every path below is relative to the notebooks/ folder:  ../data   ../results/tfm_bench   tfm_bench/*.py
import os, sys, glob, shutil, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules


def secret(name):
    """Colab secret (key icon on the left) or, locally, an environment variable. None if missing."""
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/ASML"
    tok = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    if os.path.isdir(f"{REPO}/.git"):   # the Colab copy is disposable: make it match GitHub exactly
        subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "@{u}"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, REPO], check=True)
    with open(f"{REPO}/.git/info/exclude", "a") as fh:   # keep the Drive links out of git even if .gitignore says "data/"
        fh.write("\n/data\n/results\n")
    os.chdir(f"{REPO}/notebooks")
    for sub in ["data", "results"]:     # not in git: link them to Drive so ../data and ../results work
        os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
        if not os.path.islink(f"../{sub}"):
            if os.path.isdir(f"../{sub}"):
                shutil.rmtree(f"../{sub}")
            os.symlink(f"{DRIVE_ROOT}/{sub}", f"../{sub}")
else:
    assert os.path.isdir("tfm_bench"), "run this notebook from the repo's notebooks/ folder"

DATA, WORK = os.path.abspath("../data"), os.path.abspath("../results/tfm_bench")
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)


def restore_state(zip_path, work_dir, data_dir):
    """Unzip a state zip: data CSVs -> data_dir (only if missing), results*/ folders -> work_dir.
    Handles zips whose content sits under a folder (e.g. project/), like the old Kaggle ones."""
    tmp = os.path.abspath("../results/_state_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(tmp)
    hits = glob.glob(f"{tmp}/**/results*/*/results_*/predictions.csv", recursive=True)
    root = min((h.rsplit("/", 4)[0] for h in hits), key=len) if hits else tmp
    for sub in sorted(os.path.basename(p) for p in glob.glob(f"{root}/results*") if os.path.isdir(p)):
        shutil.copytree(f"{root}/{sub}", f"{work_dir}/{sub}", dirs_exist_ok=True)
    if os.path.exists(f"{root}/context_selection.csv"):
        shutil.copy(f"{root}/context_selection.csv", work_dir)
    for p in glob.glob(f"{tmp}/**/opt_results_*.csv", recursive=True):
        if not glob.glob(f"{data_dir}/**/{os.path.basename(p)}", recursive=True):
            shutil.copy(p, data_dir)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"restored {os.path.basename(zip_path)}: {len(glob.glob(f'{work_dir}/results*/*/results_*/predictions.csv'))} "
          f"results, {len(glob.glob(f'{data_dir}/**/opt_results_*.csv', recursive=True))} data CSVs")


print("notebooks dir:", os.getcwd(), "| data:", DATA, "| results:", WORK)

if STATE_ZIP and os.path.exists(STATE_ZIP) and not glob.glob(f"{DATA}/**/opt_results_*.csv", recursive=True):
    restore_state(STATE_ZIP, WORK, DATA)

In [ ]:
# Install the model libraries (pinned to the versions the saved results used) + pass the keys to the scripts.
%pip install -q "causilo==1.0.3" "structured-data-models==0.2.1" "tabpfn==9.2.0" "botorch==0.18.1" "gpytorch==1.15.2"
for k in ["TABPFN_TOKEN", "HF_TOKEN"]:
    v = secret(k)
    if v:
        os.environ[k] = v
    else:
        print(f"(no {k})")
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"
!nvidia-smi -L

In [ ]:
# Smoke test: every model fits 50 synthetic rows and predicts 10 (also downloads all weights). Expect sorted=True.
import gc, importlib, numpy as np, torch
sys.path.insert(0, os.path.abspath("tfm_bench"))
import config as C, models; importlib.reload(C); importlib.reload(models)
print("datasets found:", len(C.DATASETS))
rng = np.random.default_rng(0)
X = rng.normal(size=(60, 9)); y = X[:, 0] - X[:, 1] ** 2 + 0.1 * rng.normal(size=60)
k = np.round(70 / (1 + np.exp(-y))); n = np.full(60, 70.0)
names = (C.GP_MODELS if GPS == "all" else [g for g in GPS.split(",") if g]) + (C.TFM_MODELS if TFMS == "all" else TFMS.split(","))
for name in names:
    p = models.FACTORIES[name]()(X[:50], y[:50], k[:50], n[:50], np.empty((0, 9)), np.empty(0), X[50:])
    print(f"{name:12s} sorted={bool((np.diff(p['quantiles'], axis=1) >= 0).all())} mean[0]={p['mean'][0]:.3f}")
gc.collect(); torch.cuda.empty_cache()

## Run
GPs are CPU-only; TFMs use the GPU. Each cell skips (config, dataset) pairs that are already done.

In [ ]:
steps = f"--max-steps {MAX_STEPS}" if MAX_STEPS else ""
for out in OUTPUTS:   # GPs. For the regressor outputs only gpr_botorch applies.
    gps = GPS if out == "dy" else ",".join(g for g in (["gpr_botorch"] if GPS == "all" else GPS.split(",")) if g and g != "gpr_laplace")
    if gps:
        !python tfm_bench/run_gpr.py --target {out} --models {gps} --datasets {DATASETS} --skip-existing {steps}

In [ ]:
for out in OUTPUTS:   # TFMs
    tfms = TFMS if out == "dy" else ",".join(m for m in TFMS.split(",") if m != "causilo_trials")
    if tfms:
        !python tfm_bench/run_tfm.py --target {out} --models {tfms} --arms {ARMS} --datasets {DATASETS} --skip-existing {steps}

## Reports + save
Rebuilt from the saved predictions (no model reruns), written to `results/tfm_bench/avg_everything/`:
`binomial_comparison.html` (die yield as counts, raw), `targets_comparison.html` (regressor outputs, normalization
switchable), `summary.html` (original yield summary). All self-contained: open them in any browser.

In [ ]:
!python tfm_bench/binomial_report.py
!python tfm_bench/targets_report.py --norm robust_z
!python tfm_bench/analyze.py

In [ ]:
# slim state zip in results/tfm_bench/: data + predictions (everything else is recomputed by the report scripts)
import time
name = f"{WORK}/benchmark_state_{time.strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{DATA}/**/opt_results_*.csv", recursive=True):
        z.write(p, "data/" + os.path.basename(p))
    for p in glob.glob(f"{WORK}/results*/*/results_*/predictions.csv") + glob.glob(f"{WORK}/results*/*.json") + \
             glob.glob(f"{WORK}/context_selection.csv"):
        z.write(p, os.path.relpath(p, WORK))
print(f"saved {name} ({os.path.getsize(name) / 1e6:.0f} MB)")